# RNA3D Phase 1 — RhoFold+ 10-target single-sequence pilot & resumable batch

**Goal:** Run *exactly the same 10 frozen RNA target FASTA sequences* used by the NuFold and DRfold2 pilots; preserve verified predictions on Google Drive. This notebook runs **inference only**. Native structures are **never supplied** to RhoFold+.

**Important experimental label:** `RhoFold+ / single_seq_pred=True / relax_steps=1000` is a **sequence-only exploratory condition**, **not** the full MSA-based RhoFold+ primary benchmark. At the time this notebook was prepared, the Drive MSA availability table recorded depth 1 for 9/10 targets and depth 4 for `9fia_bJ`. The upstream repository explicitly identifies its sequence-only mode as a quick-reference/testing pathway; an MSA-based benchmark requires a separately standardized MSA search and provenance policy.

**Run order:** Mount Drive → settings → preflight inputs → pinned source → environment → checkpoint → runtime verification → reusable inference engine → **pilot** `8ubd_H` → inspect → explicitly enable **batch**. A completed target is not recomputed if its completion record and PDB checksum validate.

**Upstream pinned source:** https://github.com/ml4bio/RhoFold/tree/6bdfbda720184409eb682ce08c05d258162ddc48  
**Historical smoke test:** `3owzA` with FASTA+A3M, Tesla T4, Python 3.7.12, PyTorch 1.10.2, ~879 s wall time including 1000-step Amber relaxation. **A 10-target sequence-only inference run has *not yet* been verified.**

**Storage:** Only this new run's files are written beneath `/MyDrive/RNA3D_Benchmark/RhoFold/runs/`. The notebook never edits the project GitHub repository or earlier NuFold/DRfold2 outputs.


## 1. Mount the Google Drive holding the existing benchmark data

Use the same Google account containing the `RNA3D_Benchmark` folder and the previous DRfold2 pilot.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Frozen experimental configuration

**Do not change `RUN_ID`, `RELAX_STEPS`, `TARGETS` or the input-mode label after predictions have been saved.** A different condition needs a new run ID. `RELAX_STEPS=1000` reproduces the *relaxation setting* of the verified RhoFold+ smoke test; it is not a claim that the sequence-only condition has been validated.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, time, datetime, math, csv, tempfile, threading

ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
NUFOLD_RUN = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
DRFOLD2_RUN = ROOT/'DRfold2/runs/drfold2_pilot10_20261110_v1'
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
PILOT_TARGET = '8ubd_H'
RUN_ID = 'rhofoldplus_pilot10_single_relax1000_20261111_v1'
MODE = 'single_sequence_quick_reference' # NEVER treat as a genuine-MSA benchmark
RELAX_STEPS = 1000
PINNED_COMMIT = '6bdfbda720184409eb682ce08c05d258162ddc48'
REPO = Path('/content/RhoFold')
MAMBA = Path('/content/bin/micromamba')
MAMBA_ROOT = Path('/content/micromamba_root')
CHECKPOINT = REPO/'pretrained/RhoFold_pretrained.pt'
CKPT_URL = 'https://huggingface.co/cuhkaih/rhofold/resolve/main/rhofold_pretrained_params.pt'
RUN_ROOT = ROOT/'RhoFold/runs'/RUN_ID
WORK_ROOT = Path('/content/rna3d_rhofoldplus_pilot_work')
REQUIRED_SOURCE = ('relaxed_1000_model.pdb' if RELAX_STEPS == 1000 else
                   ('unrelaxed_model.pdb' if RELAX_STEPS == 0 else f'relaxed_{RELAX_STEPS}_model.pdb'))
assert MODE == 'single_sequence_quick_reference'
assert RELAX_STEPS == 1000, 'This frozen experiment uses 1000-step relaxation; use a NEW run ID for other conditions.'
assert len(TARGETS) == 10 and len(set(TARGETS)) == 10
print('Experiment:',RUN_ID,'| Mode:',MODE,'| Relax steps:',RELAX_STEPS)
print('Output root:',RUN_ROOT)


## 3. Input integrity and storage preflight (no GPU work)

Verify each FASTA against the completed DRfold2 record. This ensures we compare **the same biological input sequences**. Native 3D coordinates are not loaded at any inference step.


In [ ]:
def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as inp:
        for chunk in iter(lambda: inp.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

def fasta_sequence(path):
    raw=path.read_text().splitlines()
    lines=[l.strip() for l in raw if l.strip()]
    if sum(l.startswith('>') for l in lines)!=1 or not lines[0].startswith('>'):
        raise ValueError(f'Expected one FASTA record: {path}')
    seq=''.join(lines[1:]).upper().replace('T','U')
    if not seq or set(seq)-set('ACGU'):
        raise ValueError(f'Unexpected sequence content: {path}')
    return seq

assert ROOT.is_dir(), 'Drive not mounted or RNA3D_Benchmark folder missing'
assert NUFOLD_RUN.is_dir() and DRFOLD2_RUN.is_dir(), 'Expected source run missing; check Google account'
FASTA={}; SEQS={}; FASTA_HASH={}
for target in TARGETS:
    path=NUFOLD_RUN/'inputs'/target/f'{target}.fasta'
    marker=DRFOLD2_RUN/target/'complete.json'
    if not path.is_file() or not marker.is_file():
        raise FileNotFoundError(f'{target}: missing original FASTA or DRfold2 completion record')
    prior=json.loads(marker.read_text())
    if prior.get('status')!='complete' or prior.get('target_id')!=target:
        raise RuntimeError(f'{target}: invalid DRfold2 completion marker')
    h=sha256_file(path)
    if h!=prior['fasta_sha256']:
        raise RuntimeError(f'{target}: FASTA SHA-256 does not match previous DRfold2 input')
    FASTA[target], SEQS[target], FASTA_HASH[target]=path,fasta_sequence(path),h
    print(f'{target:10} | {len(SEQS[target]):>3} nt | FASTA matches DRfold2: YES')
print('INPUT PREFLIGHT PASS: 10/10; experimental native PDBs were not read')


### GPU availability gate — run before installing old dependencies

**Stop here if Colab GPU quota is exhausted.** This preflight avoids downloading old Conda packages and ~485 MiB checkpoint when no GPU is connected.


In [ ]:
gpu_probe=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True)
if gpu_probe.returncode!=0 or not gpu_probe.stdout.strip():
    raise RuntimeError('No usable GPU runtime detected. Change Colab runtime to GPU and rerun from the top when GPU access returns. Do not install the large RhoFold environment yet.')
print('Pre-install GPU gate PASS:',gpu_probe.stdout.strip().splitlines()[0])


## 5. Clone and pin the **official** RhoFold+ source

The GitHub repository is downloaded locally in ephemeral Colab storage. **No push or modification to the FYP repository.** The exact upstream commit is verified before running anything.


In [ ]:
def run_checked(args, cwd=None, timeout=None):
    print('$', ' '.join(map(str,args)),flush=True)
    subprocess.run([str(x) for x in args],cwd=cwd,check=True,timeout=timeout)

if not REPO.is_dir():
    run_checked(['git','clone','https://github.com/ml4bio/RhoFold.git',str(REPO)])
assert (REPO/'.git').is_dir(), f'Not a Git repository: {REPO}'
observed=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if observed != PINNED_COMMIT:
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    if dirty:
        raise RuntimeError('Existing RhoFold checkout has local modifications; refusing to change checkout')
    run_checked(['git','-C',str(REPO),'checkout','--detach',PINNED_COMMIT])
observed=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert observed == PINNED_COMMIT, f'Unexpected upstream revision: {observed}'
assert (REPO/'inference.py').is_file() and (REPO/'envs/environment_linux.yaml').is_file()
print('Pinned official RhoFold+ commit verified:', observed)


## 6. Install the previously verified Micromamba environment

The upstream environment is old (Python 3.7 / PyTorch 1.10.2 / CUDA 11.3); Colab solver/package availability may change. Do **not** silently replace pinned dependencies with current Colab PyTorch. If environment creation fails, preserve the error and ask for help before continuing.


In [ ]:
if not MAMBA.is_file():
    run_checked(['bash','-lc',
        'set -euo pipefail; curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'])
assert MAMBA.is_file()
ENV_PY = MAMBA_ROOT/'envs/rhofold/bin/python'
if not ENV_PY.is_file():
    run_checked([MAMBA,'-r',MAMBA_ROOT,'create','-y','-n','rhofold','-f',REPO/'envs/environment_linux.yaml'])
assert ENV_PY.is_file(), 'Expected RhoFold+ environment python missing'
# setup.py install matches the successful historical Colab smoke test.
run_checked([MAMBA,'-r',MAMBA_ROOT,'run','-n','rhofold','python','setup.py','install'],cwd=REPO)
print('RhoFold+ environment + package installation: complete')


## 7. Download/check the official model checkpoint

Expected historical checkpoint size is approximately **485 MiB**. Record its **actual SHA-256** rather than assuming it has never changed upstream. RhoFold's unmodified `inference.py` also calls Hugging Face `snapshot_download` at startup; the first inference requires network access.


In [ ]:
CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)
if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size < 300*1024*1024:
    if CHECKPOINT.exists():
        raise RuntimeError('A truncated checkpoint exists; inspect it before retrying, do not overwrite blindly')
    part=CHECKPOINT.with_suffix('.pt.part')
    if part.exists():
        raise RuntimeError('Partial checkpoint download exists; inspect/remove that partial temporary file manually')
    try:
        run_checked(['wget','-q','--show-progress','-O',part,CKPT_URL],timeout=1800)
        if part.stat().st_size < 300*1024*1024:
            raise RuntimeError('Checkpoint unexpectedly small')
        os.replace(part,CHECKPOINT)
    except Exception:
        print('Download unsuccessful; partial file retained for inspection:',part)
        raise
CKPT_HASH=sha256_file(CHECKPOINT)
print('Checkpoint:',CHECKPOINT,'| Bytes:',CHECKPOINT.stat().st_size)
print('Checkpoint SHA-256:',CKPT_HASH)


## 8. Verify GPU and CLI before inference

**GPU runtime required** for this inference stage. If Google Colab has reached its GPU quota, do not run the following pilot until a supported GPU is available; CPU is reserved for later evaluation.


In [ ]:
verify_script = "import sys, torch, numpy; print('Python',sys.version.split()[0]); print('Torch',torch.__version__); print('NumPy',numpy.__version__); print('CUDA',torch.cuda.is_available()); print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); assert torch.cuda.is_available(), 'GPU required'"
run_checked([MAMBA,'-r',MAMBA_ROOT,'run','-n','rhofold','python','-c',verify_script],cwd=REPO)
help_test=subprocess.run([str(MAMBA),'-r',str(MAMBA_ROOT),'run','-n','rhofold','python','inference.py','--help'],cwd=REPO,capture_output=True,text=True)
assert help_test.returncode==0, f'CLI import failed:\n{help_test.stderr[-1800:]}'
for flag in ['--input_fas','--output_dir','--single_seq_pred','--relax_steps','--ckpt']:
    assert flag in help_test.stdout, f'CLI flag missing: {flag}'
GPU_NAME=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).strip().splitlines()[0]
TORCH_VERSION=subprocess.check_output([str(MAMBA),'-r',str(MAMBA_ROOT),'run','-n','rhofold','python','-c','import torch; print(torch.__version__)'],text=True).strip().splitlines()[-1]
print('GPU preflight PASS:',GPU_NAME,'| torch:',TORCH_VERSION)


## 9. Inference engine — immutable artifacts, validation, automatic skip

- Performs **one target at a time** in `/content/rna3d_rhofoldplus_pilot_work/`.
- Calls upstream `inference.py --input_fas <fasta> --single_seq_pred True --relax_steps 1000` **without an MSA or native structure**.
- Requires `relaxed_1000_model.pdb` for the predeclared primary output. It never silently substitutes an unrelaxed PDB.
- Saves output artifacts and completion metadata to **Google Drive**, with `complete.json` written last.
- On restart, checks input hash + checkpoint hash + source commit + output PDB hash + PDB sequence and skips only fully verified targets.
- If an interrupted/failed target folder exists, stops for manual inspection rather than overwriting it.
- Measures observed GPU memory via `nvidia-smi` polling; this is approximate (not an isolated GPU allocation).


In [ ]:
RNA_CODES={'A':'A','C':'C','G':'G','U':'U','RA':'A','RC':'C','RG':'G','RU':'U',
           'ADE':'A','CYT':'C','GUA':'G','URA':'U'}

def verify_pdb(path, expected_sequence):
    if not path.is_file() or path.stat().st_size<150:
        raise ValueError(f'Missing or tiny PDB: {path}')
    seen=set(); residue_order=[]; atoms=0; chains=set()
    for line in path.read_text(errors='replace').splitlines():
        if not line.startswith(('ATOM  ','HETATM')):
            continue
        if len(line)<54: raise ValueError('Truncated coordinate record')
        if line[16:17] not in (' ','A','1'): continue
        key=(line[21:22],line[22:26],line[26:27],line[17:20].strip().upper())
        for a,b in ((30,38),(38,46),(46,54)):
            if not math.isfinite(float(line[a:b])):
                raise ValueError('Non-finite coordinate')
        atoms+=1; chains.add(key[0])
        if key not in seen:
            seen.add(key)
            residue_order.append(RNA_CODES.get(key[3],'N'))
    seq=''.join(residue_order)
    if not atoms or seq!=expected_sequence:
        raise ValueError(f'PDB sequence mismatch: {path.name} has {seq}; expected {expected_sequence}')
    if len(chains)!=1:
        raise ValueError(f'Expected one RNA chain; found {chains}')
    return {'residues':len(seq),'atom_records':atoms,'size_bytes':path.stat().st_size,'chain':next(iter(chains))}

def atomic_json(path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile('w',dir=path.parent,prefix='tmp_',suffix='.json',delete=False) as f:
        tmp=Path(f.name); json.dump(obj,f,indent=2,sort_keys=True); f.write('\n')
    try: os.replace(tmp,path)
    finally:
        if tmp.exists(): tmp.unlink()

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def complete_marker(target):
    return RUN_ROOT/target/'complete.json'

def verified_complete(target):
    path=complete_marker(target)
    if not path.is_file(): return False
    try:
        info=json.loads(path.read_text())
        pdb=RUN_ROOT/target/info['primary_pdb_name']
        conditions=(info.get('status')=='complete' and info.get('target_id')==target
            and info.get('model_commit')==PINNED_COMMIT
            and info.get('checkpoint_sha256')==CKPT_HASH
            and info.get('fasta_sha256')==FASTA_HASH[target]
            and info.get('input_mode')==MODE and info.get('relax_steps')==RELAX_STEPS
            and info.get('primary_pdb_name')==REQUIRED_SOURCE
            and sha256_file(pdb)==info.get('primary_pdb_sha256'))
        if not conditions: return False
        verify_pdb(pdb,SEQS[target])
        return True
    except Exception:
        return False

def nvidia_used_mib():
    try:
        raw=subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],
                                    text=True,timeout=8).strip().splitlines()[0]
        return int(raw)
    except Exception:
        return None

def execute_target(target):
    if target not in TARGETS: raise ValueError('Target not in frozen list')
    dest=RUN_ROOT/target
    if verified_complete(target):
        print('SKIP verified complete:',target,flush=True)
        return 'skipped'
    if dest.exists():
        raise FileExistsError(f'{target}: existing output is incomplete or checksum/provenance mismatched: {dest}. Inspect, do NOT delete/overwrite blindly.')
    scratch=WORK_ROOT/target
    if scratch.exists():
        raise FileExistsError(f'{target}: nonempty/previous local scratch exists: {scratch}. Inspect before retry.')
    scratch.mkdir(parents=True,exist_ok=False)
    out=scratch/'prediction'; out.mkdir()
    log=scratch/'inference.log'
    command=[str(MAMBA),'-r',str(MAMBA_ROOT),'run','-n','rhofold','python','inference.py',
             '--input_fas',str(FASTA[target]),'--single_seq_pred','True',
             '--output_dir',str(out),'--ckpt',str(CHECKPOINT),
             '--relax_steps',str(RELAX_STEPS),'--device','cuda:0']
    print(f'START {target} ({len(SEQS[target])} nt) | {MODE} | relax={RELAX_STEPS}',flush=True)
    stop=threading.Event(); mem=[]
    def sampler():
        while not stop.wait(2.0):
            x=nvidia_used_mib()
            if x is not None: mem.append(x)
    monitor=threading.Thread(target=sampler,daemon=True)
    monitor.start(); start=time.monotonic()
    try:
        with log.open('w') as f:
            # stderr and stdout saved together, so errors remain inspectable in Drive.
            proc=subprocess.run(command,cwd=REPO,stdout=f,stderr=subprocess.STDOUT,timeout=7200,check=False)
        seconds=round(time.monotonic()-start,3)
        if proc.returncode!=0:
            raise RuntimeError(f'Upstream inference exited {proc.returncode}; log tail:\n'+log.read_text(errors='replace')[-1600:])
        if sha256_file(CHECKPOINT)!=CKPT_HASH:
            raise RuntimeError('Checkpoint hash changed during inference; refusing to finalize this run')
        primary=out/REQUIRED_SOURCE
        validation=verify_pdb(primary,SEQS[target])
        secondary=out/'unrelaxed_model.pdb'
        verify_pdb(secondary,SEQS[target])
        # Copy only AFTER verifying all needed model outputs; complete marker is written LAST.
        dest.mkdir(parents=True,exist_ok=False)
        files=[REQUIRED_SOURCE,'unrelaxed_model.pdb','log.txt','ss.ct','results.npz']
        for name in sorted(set(files)):
            source=out/name
            if not source.is_file() or source.stat().st_size==0:
                raise FileNotFoundError(f'Required upstream output missing: {source}')
            shutil.copy2(source,dest/name)
            if sha256_file(dest/name)!=sha256_file(source):
                raise RuntimeError(f'Output copy checksum mismatch: {name}')
        shutil.copy2(log,dest/'inference.log')
        if sha256_file(dest/'inference.log')!=sha256_file(log):
            raise RuntimeError('Log copy checksum mismatch')
        record={
            'status':'complete','target_id':target,'model':'RhoFold+',
            'model_commit':PINNED_COMMIT,'input_mode':MODE,
            'single_seq_pred':True,'msa_depth':1,'input_msa':None,
            'fasta_sha256':FASTA_HASH[target],'checkpoint_sha256':CKPT_HASH,
            'relax_steps':RELAX_STEPS,'primary_pdb_name':REQUIRED_SOURCE,
            'primary_pdb_sha256':sha256_file(dest/REQUIRED_SOURCE),
            'unrelaxed_pdb_sha256':sha256_file(dest/'unrelaxed_model.pdb'),
            'input_length':len(SEQS[target]),'pdb_validation':validation,
            'seconds':seconds,'gpu':GPU_NAME,'torch_version':TORCH_VERSION,
            'gpu_memory_used_mib_peak_observed':max(mem) if mem else None,
            'timestamp_utc':utc_now(),'command':command,
            'note':'Sequence-only quick-reference/ablation condition, NOT full MSA benchmark.'}
        atomic_json(dest/'complete.json',record)
        assert verified_complete(target), f'Post-write verification failed: {target}'
        print(f'COMPLETE {target} seconds={seconds} residues={validation["residues"]} saved_to={dest}',flush=True)
        return 'complete'
    except Exception as err:
        dest.mkdir(parents=True,exist_ok=True)
        if log.is_file() and not (dest/'inference.log').exists():
            shutil.copy2(log,dest/'inference.log')
        atomic_json(dest/'failed.json',{'status':'failed_or_interrupted','target_id':target,
                    'timestamp_utc':utc_now(),'error':repr(err),'command':command,
                    'input_mode':MODE,'relax_steps':RELAX_STEPS})
        print('ERROR:',target,repr(err),'| diagnostic files:',dest,flush=True)
        raise
    finally:
        stop.set(); monitor.join(timeout=3)

print('Inference engine prepared. Neither pilot nor batch has run yet.')


## 10. PILOT MODE — `8ubd_H` **only**

The 36-nt pilot checks whether this **sequence-only mode** actually works with the pinned upstream checkpoint and environment. A previous smoke test on `3owzA` used a genuine A3M, so it does not prove this path yet.


In [ ]:
pilot_status=execute_target(PILOT_TARGET)
print('Pilot status:',pilot_status)
assert verified_complete(PILOT_TARGET)
print('Pilot complete record:',complete_marker(PILOT_TARGET))


## 11. Review pilot provenance and log before batch

Confirm that `single_seq_pred=True`, predicted residues match the FASTA, and the expected *relaxed* PDB was saved. Do not enable batch if log contains unexplained errors.


In [ ]:
p=RUN_ROOT/PILOT_TARGET
print((p/'complete.json').read_text()[:4000])
print('\nINFERENCE LOG (last ~45 lines):\n')
print('\n'.join((p/'inference.log').read_text(errors='replace').splitlines()[-45:]))
print('\nPilot verification:',verified_complete(PILOT_TARGET))


## 12. BATCH MODE — manual opt-in

After reviewing a successful pilot, set `RUN_BATCH=True` **in this cell**. It checks the pilot's completed marker and skips completed targets. It stops immediately on any failed/incomplete result rather than risking repeated inference or overwriting evidence.


In [ ]:
RUN_BATCH = False  # Set True ONLY after pilot review
if not RUN_BATCH:
    print('Batch disabled. No additional targets will run.')
else:
    assert verified_complete(PILOT_TARGET), 'Review/verify pilot before enabling batch'
    for target in TARGETS:
        print('\n----',target,'----',flush=True)
        execute_target(target)
    count=sum(verified_complete(t) for t in TARGETS)
    print('VERIFIED COMPLETE:',count,'/',len(TARGETS))
    assert count==10


## 13. Read-only status & lightweight inference summary

If Colab disconnects after a completed run, reconnect and remount Drive. To **inspect saved status only**, this cell does not need model setup, a checkpoint, or GPU. If 10/10 are present, later build a **separate CPU-only structural evaluation** using the NuFold/DRfold2 native mapping + pinned US-align protocol.


In [ ]:
# Drive-only audit: inspect existing completion records and output file presence.
fields=['target_id','status','seconds','input_length','relax_steps','primary_pdb_name','model_commit','checkpoint_sha256','primary_pdb_sha256']
records=[]
for target in TARGETS:
    folder=RUN_ROOT/target
    marker=folder/'complete.json'
    if marker.is_file():
        try:
            r=json.loads(marker.read_text())
            pdb=folder/r.get('primary_pdb_name','MISSING')
            valid=(r.get('status')=='complete' and r.get('target_id')==target
                and r.get('input_mode')==MODE and r.get('relax_steps')==RELAX_STEPS
                and pdb.is_file() and sha256_file(pdb)==r.get('primary_pdb_sha256')
                and r.get('fasta_sha256')==FASTA_HASH.get(target))
            status='COMPLETE/checksums_pass' if valid else 'INVALID/check_manually'
            records.append({**r,'status':status})
        except Exception as e:
            records.append({'target_id':target,'status':f'INVALID ({e})'})
    else:
        status='FAILED/incomplete' if folder.exists() else 'NOT RUN'
        records.append({'target_id':target,'status':status})
for r in records:
    print(f"{r['target_id']:10} {r['status']:23} {str(r.get('seconds','')):>8} s")
print('Saved and checksum-validated:',sum(r['status']=='COMPLETE/checksums_pass' for r in records),'/',len(TARGETS))
print('Drive run:',RUN_ROOT)
print('Accuracy evaluation: NOT RUN; these are inference records only.')


## Failure / resume guidance

1. If the runtime disconnects **between** targets, reconnect, mount Drive, set the **same** `RUN_ID` and model mode, reinstall ephemeral environment/checkpoint, then rerun the pilot/batch cells. Verified completed targets are skipped automatically.
2. If the runtime stops **during a target**, its Drive folder may contain `failed.json`/partial files. The engine deliberately **refuses to overwrite**. Inspect `<target>/inference.log` and `failed.json`, then either archive that failed folder before retrying or use a **new experiment/run ID** (without mixing outcomes under one name). Do not delete successful target folders.
3. If Hugging Face replaces the checkpoint, the SHA-256 mismatch prevents an invalid resume. Reuse the exact checkpoint bytes or start a clearly separated run.
4. A standard RhoFold+ MSA-based benchmark needs a common external MSA-generation method with database versions, cutoffs, per-target depth, and separately frozen inputs. Do not call this sequence-only pilot the MSA-based result.
5. For comparison with NuFold/DRfold2, use the **same native observed-residue positions** and `USalign -mol RNA -TMscore 1` normalized by experimental native. Preserve the chosen RhoFold relaxed-output policy in all reports.

**No GitHub writes are performed by this notebook.**
